# RoPE with a key cache in NumPy

A decoder generates text one token at a time. With RoPE, each new key is rotated once, at
its own position, and stored in the cache already rotated. Later queries use the cached keys
as they are, and the scores still depend only on how far apart the tokens are. You will write
the rotation for a batch of vectors, each at its own position, and then a decoding loop that
keeps such a cache.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Twelve tokens of one attention head of width 8: queries, keys and values.
rng = np.random.default_rng(11)
T, d = 12, 8
Q, K, V = rng.standard_normal((3, T, d))
print(f"{T} tokens, head width {d}")

## Part 1 (coding task): Rotate a batch at its positions

Write `rope(X, pos, base=10000.0)`. `X` has shape `(n, d)` and `pos` has shape `(n,)`.
Row `t` is rotated on each pair of consecutive coordinates `(2i, 2i + 1)`, the layout the
lesson uses, by the angle $\alpha = \text{pos}[t]\,\theta_i$ with
$\theta_i = \text{base}^{-2i/d}$:

$$
(x_{2i},\ x_{2i+1}) \mapsto (x_{2i}\cos\alpha - x_{2i+1}\sin\alpha,\ \ x_{2i}\sin\alpha + x_{2i+1}\cos\alpha).
$$

Return a new array and leave `X` as it was. Positions can be any numbers, not only
`0, 1, 2, ...`.

In [ ]:
def rope(X, pos, base=10000.0):
    """Rotate row t of X by pos[t] * theta_i on each coordinate pair (2i, 2i + 1)."""
    X = np.asarray(X, dtype=float)
    pos = np.asarray(pos, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
one = rope(np.array([[1.0, 0.0, 0.0, 1.0]]), np.array([2.0]))
assert np.shape(one) == (1, 4), f"one row of width 4 came back with shape {np.shape(one)}"
assert not np.allclose(one[0, 2:], [-np.sin(0.2), np.cos(0.2)]), "pair 1 turned by 0.2: theta_i is base ** (-2i / d), so with d = 4 pair 1 turns 0.01 per position"
pair0, pair1 = np.exp(2j), 1j * np.exp(0.02j)          # (1, 0) turned by 2 and (0, 1) turned by 0.02
want_one = [pair0.real, pair0.imag, pair1.real, pair1.imag]
assert np.allclose(one, [want_one]), f"one row at position 2 gave {np.round(one, 4)}, expected {np.round(want_one, 4)}: the pairs are coordinates (0, 1) and (2, 3), and each turns counterclockwise"
assert np.allclose(rope(np.array([[1.0, 0.0, 1.0, 0.0]]), np.array([1.0]), base=100.0), [[np.cos(1), np.sin(1), np.cos(0.1), np.sin(0.1)]]), "with base=100 pair 1 should turn 0.1 per position: use the base argument"

X4 = np.arange(32.0).reshape(4, 8) / 10
X4_before = X4.copy()
pos4 = np.array([0.0, 1.0, 5.0, 2.5])
z = (X4[:, 0::2] + 1j * X4[:, 1::2]) * np.exp(1j * pos4[:, None] * 10000.0 ** (-np.arange(4) / 4))
want4 = np.empty_like(X4)
want4[:, 0::2], want4[:, 1::2] = z.real, z.imag
got4 = rope(X4, pos4)
assert np.array_equal(X4, X4_before), "rope changed its input X: write the result into a new array"
assert np.allclose(got4, want4), "four rows of width 8, each at its own position, came out wrong. Every row needs its own position times every pair's theta, an (n, d/2) table of angles"

S0 = rope(Q, np.arange(T)) @ rope(K, np.arange(T)).T
S1 = rope(Q, np.arange(T) + 37.0) @ rope(K, np.arange(T) + 37.0).T
assert np.allclose(S0, S1), "moving every position along by 37 changed the scores: they should depend on offsets only"
print("Part 1 passed")

## Part 2 (coding task): Decode with a cache of rotated keys

Write `decode(Q, K, V, start=0)`: causal attention computed one token at a time, the way a
model generates. Token `t` sits at position `start + t`. At step `t`:

1. Rotate key `t` at its position and append it to the cache. Append value `t` as it is.
2. Rotate query `t` at the same position.
3. Score it against every cached key, divide by $\sqrt{d}$, take a softmax over the cache,
   and return the weighted sum of the cached values.

Rotate each key exactly once, in the step that adds it, and never rotate a value. Use your
`rope` from Part 1 for every rotation, since the check counts the rows it rotates. Return
the `T` outputs as one array of shape `(T, d)`.

In [ ]:
def decode(Q, K, V, start=0):
    """Causal attention, one step per token, with a cache of rotated keys."""
    # TODO
    raise NotImplementedError

In [ ]:
def causal_reference(Q, K, V, turn_values=False, scale=True, rotate=True):
    """Full causal attention with RoPE written with complex numbers, and some common slips."""
    n, dd = Q.shape
    theta = 10000.0 ** (-2.0 * np.arange(dd // 2) / dd)
    turn = np.exp(1j * np.arange(n)[:, None] * theta) if rotate else np.ones((n, dd // 2))
    zq = (Q[:, 0::2] + 1j * Q[:, 1::2]) * turn
    zk = (K[:, 0::2] + 1j * K[:, 1::2]) * turn
    S = (zq @ zk.conj().T).real / (np.sqrt(dd) if scale else 1.0)
    S = np.where(np.tri(n, dtype=bool), S, -np.inf)
    A = np.exp(S - S.max(axis=1, keepdims=True))
    A /= A.sum(axis=1, keepdims=True)
    if turn_values:
        zv = (V[:, 0::2] + 1j * V[:, 1::2]) * turn
        Vt = np.empty_like(V)
        Vt[:, 0::2], Vt[:, 1::2] = zv.real, zv.imag
        return A @ Vt
    return A @ V

rows_rotated = []
part1_rope = rope
def rope(X, pos, base=10000.0):
    X = np.asarray(X, dtype=float)
    rows_rotated.append(X.size // X.shape[-1])
    return part1_rope(X, pos, base)
try:
    out = np.asarray(decode(Q, K, V))
finally:
    rope = part1_rope

assert out.shape == (T, d), f"decode returned shape {out.shape}, expected ({T}, {d})"
want = causal_reference(Q, K, V)
slips = [
    (causal_reference(Q, K, V, rotate=False), "the outputs do not depend on position at all. That happens when every cached key is turned to the query's position at each step: each key must keep the rotation of its own position"),
    (causal_reference(Q, K, V, turn_values=True), "the values were rotated: RoPE turns queries and keys only"),
    (causal_reference(Q, K, V, scale=False), "the scores were not divided by sqrt(d)"),
]
if not np.allclose(out, want):
    hint = next((message for wrong, message in slips if np.allclose(out, wrong)), "the outputs differ from causal attention with RoPE")
    first = int(np.argmax(~np.isclose(out, want).all(axis=1)))
    raise AssertionError(f"{hint} (first wrong row: token {first})")
assert sum(rows_rotated) == 2 * T, f"rope rotated {sum(rows_rotated)} rows for {T} tokens: rotate each query and each key once, {2 * T} rows in all, and keep the rotated keys in the cache instead of rotating them again"
assert np.allclose(decode(Q, K, V, start=500), want), "starting the text at position 500 changed the outputs, but the scores should depend on offsets only"
print("Part 2 passed")

## Solution

### Part 1: Rotate a batch at its positions

In [ ]:
def rope(X, pos, base=10000.0):
    """Rotate row t of X by pos[t] * theta_i on each coordinate pair (2i, 2i + 1)."""
    X = np.asarray(X, dtype=float)
    pos = np.asarray(pos, dtype=float)
    d = X.shape[1]
    theta = base ** (-2.0 * np.arange(d // 2) / d)
    angle = pos[:, None] * theta[None, :]          # (n, d/2): every row against every pair
    c, s = np.cos(angle), np.sin(angle)
    out = np.empty_like(X)
    out[:, 0::2] = X[:, 0::2] * c - X[:, 1::2] * s
    out[:, 1::2] = X[:, 0::2] * s + X[:, 1::2] * c
    return out

**Why it works.** `pos[:, None] * theta[None, :]` makes one angle for every row and every
pair, shape `(n, d/2)`, and the slices `0::2` and `1::2` pick the first and second
coordinate of every pair at once. Without the `[:, None]`, a `(n,)` array of positions
times a `(d/2,)` array of speeds fails for most shapes, and when `n = d/2`, as in the
check's four rows of width 8, it silently multiplies position `t` by speed `t` only. The
last check is the lesson's Toeplitz property: shifting every position by 37 leaves every
score unchanged.

### Part 2: Decode with a cache of rotated keys

In [ ]:
def decode(Q, K, V, start=0):
    """Causal attention, one step per token, with a cache of rotated keys."""
    n, d = Q.shape
    keys, values, out = [], [], []
    for t in range(n):
        position = np.array([start + t])
        keys.append(rope(K[t:t + 1], position)[0])     # rotated once, then kept
        values.append(V[t])
        q = rope(Q[t:t + 1], position)[0]
        s = np.array(keys) @ q / np.sqrt(d)
        w = np.exp(s - s.max())
        out.append(w / w.sum() @ np.array(values))
    return np.array(out)

**Why it works.** Cached key $j$ holds $R_j\mathbf{k}_j$ and query $t$ arrives as
$R_t\mathbf{q}_t$, so each pair contributes a term that depends on $j - t$ alone, exactly
as in full attention. Nothing about the cache has to know about positions: the attention
step is ordinary attention on stored vectors, and the work of RoPE is one rotation per new
query and key, $2T$ rows in all. Keeping raw keys and rotating the whole cache at every step
gives the same outputs for about $T^2/2$ rotated rows. The run with `start=500` passes for
the same reason the lesson's diagonals are constant, and a loop that used `t` instead of
`start + t` would pass too, since only differences of positions reach the scores.